# Parse FIX
Resolve classified market message rows against the FIX dictionary.

In [ ]:
project_root = "."
source = "logs.messages"
start = None
end = None
fix_dictionary = "data/fix"
null_values = ["", "null", "<null>", "n/a"]
protocols = None
fields = None
catalog = "rekep"
catalog_properties = {}
table_properties = {"history.expire.max-snapshot-age-ms": "604800000"}
branch = "root"
target_pattern = "fix.{category}"
instrument_source = "market.instruments"
instrument_snapshot_every = 3_600_000_000_000
merge_by = True
batch_row_size = 65_536
commit_row_size = 250_000
limit = None
log_level = "INFO"

In [ ]:
import pyarrow
import pyarrow.compute as pc
from pyiceberg.expressions import (
    And,
    GreaterThanOrEqual,
    In,
    IsNull,
    LessThan,
    LessThanOrEqual,
    Not,
    NotEqualTo,
    Or,
)
from rekep.enums import EventType
from rekep.fix.fields import FieldRules
from rekep.fix.registry import FixRegistry
from rekep.fix.rules import Rules
from rekep.fix.transcribe import FixCodec
from rekep.iceberg import IcebergDataset
from rekep.market import Instrument
from rekep.market.event import DAY
from rekep.text import FixMsg, Message
from rekep.times import unix_of
from rekep.urls import Url
from rekep.logs import configure

configure(log_level)

def _window(lower, upper, column="unix"):
    predicates = []
    if lower is not None:
        predicates.append(GreaterThanOrEqual(column, lower))
    if upper is not None:
        predicates.append(LessThan(column, upper))
    return (
        None
        if not predicates
        else predicates[0]
        if len(predicates) == 1
        else And(*predicates)
    )


protocol_rules = Rules() if protocols is None else Rules.from_dict(protocols)
registry = FixRegistry(
    cache_dir=Url.from_string(str(fix_dictionary)).resolve(project_root),
    offline=True,
    announce=print,
)
field_rules = FieldRules() if fields is None else FieldRules.from_dict(fields)
codec = FixCodec(
    rules=protocol_rules,
    registry=registry,
    null_values=frozenset(null_values),
    fields=field_rules,
)
field = FixMsg.into_field()
source_field = Message.into_field()
messages = IcebergDataset(
    field=source_field.with_name(source),
    catalog=catalog,
    properties=dict(catalog_properties),
    branch=branch,
)
source_columns = [
    name
    for name in (messages.table_field.names if messages.exists else source_field.names)
    if name != "message"
]
if messages.exists:
    missing = sorted(
        {"MsgType", "entries", "protocolcode"} - set(messages.table_field.names)
    )
    if missing:
        raise ValueError(
            f"{source} is missing {missing}; rebuild it with parse_messages "
            "before parsing FIX"
        )

In [ ]:
read = market_read = 0
observed_lower = observed_upper = None

# The window is read off the *stored* recording clock, because that is what
# the message stage partitioned on. `unix` moves when a transaction time
# resolves, so filtering on it here would drop rows the interval owns.
lower, upper = unix_of(start), unix_of(end, upper=True)
window = _window(lower, upper)
# The two scans partition every stored `etype`: the market scan keeps the
# compiled codes ranked at or above INTENT, and the terminal scan keeps the
# complement -- so a code no member spells still reaches the category router instead of silently matching no scan.
market_events = In("etype", EventType.ranked_at_least(EventType.INTENT))
terminal_events = Not(market_events)
market_filter = market_events if window is None else And(window, market_events)
terminal_filter = terminal_events if window is None else And(window, terminal_events)
targets = {}


def _target(category):
    target = targets.get(category)
    if target is None:
        target = targets[category] = IcebergDataset(
            field=field.with_name(target_pattern.format(category=category)),
            catalog=catalog,
            properties=dict(catalog_properties),
            table_properties=dict(table_properties),
            branch=branch,
            commit_row_size=commit_row_size,
            sort_by=("unix", "MsgSeqNum", "hash"),
        )
    return target


target = _target("market")


def _market_batches():
    global read, market_read, observed_lower, observed_upper
    for staged in messages.read_arrow_reader(
        columns=source_columns, row_filter=market_filter
    ):
        if limit is not None and read + staged.num_rows > limit:
            staged = staged.slice(0, max(0, limit - read))
        if not staged.num_rows:
            continue
        read += staged.num_rows
        market_read += staged.num_rows
        batch = FixMsg.from_message_batch(staged, codec)
        bounds = pc.min_max(batch.column("unix")).as_py()
        observed_lower = (
            bounds["min"]
            if observed_lower is None
            else min(observed_lower, bounds["min"])
        )
        observed_upper = (
            bounds["max"] + 1
            if observed_upper is None
            else max(observed_upper, bounds["max"] + 1)
        )
        yield batch
        if limit is not None and read >= limit:
            break


written = target.append_arrow_reader(
    _market_batches(),
    field,
    merge_by=merge_by,
    commit_row_size=commit_row_size,
)
skipped = market_read - written
routed = {"market": market_read}
buffers = {}
held_rows = {}


def _flush(category):
    global written, skipped
    batches = buffers.pop(category, [])
    count = held_rows.pop(category, 0)
    if not count:
        return
    landed = _target(category).append_arrow_table(
        pyarrow.Table.from_batches(batches), merge_by=merge_by
    )
    written += landed
    skipped += count - landed


for staged in messages.read_arrow_reader(
    columns=source_columns, row_filter=terminal_filter
):
    if limit is not None and read >= limit:
        break
    if limit is not None and read + staged.num_rows > limit:
        staged = staged.slice(0, max(0, limit - read))
    if not staged.num_rows:
        continue
    read += staged.num_rows
    batch = FixMsg.from_message_batch(staged, codec)
    categories = protocol_rules.into_arrow_category_array(
        batch.column("protocolcode"), batch.column("etype")
    )
    for category in sorted(pc.unique(categories).to_pylist()):
        part = batch.filter(pc.equal(categories, category))
        routed[category] = routed.get(category, 0) + part.num_rows
        buffers.setdefault(category, []).append(part)
        held_rows[category] = held_rows.get(category, 0) + part.num_rows
        if commit_row_size and held_rows[category] >= commit_row_size:
            _flush(category)
    if limit is not None and read >= limit:
        break
for category in list(buffers):
    _flush(category)

In [ ]:
lower = lower if lower is not None else observed_lower
upper = upper if upper is not None else observed_upper
instrument_table = IcebergDataset(
    field=Instrument.into_field(instrument_source),
    catalog=catalog,
    properties=dict(catalog_properties),
    table_properties=dict(table_properties),
    branch=branch,
)


def _instrument_seeds():
    if lower is None:
        return iter(())
    recent = And(
        GreaterThanOrEqual("unix", lower - DAY), LessThanOrEqual("unix", lower)
    )
    reader = instrument_table.read_arrow_reader(
        Instrument.into_field(), row_filter=recent, order_by=("unix", "version", "hash")
    )
    return Instrument.from_arrow_reader(reader)


def _market_messages():
    window = _window(lower, upper)
    # A null MsgType is not U1; `NotEqualTo` alone does not retain nulls.
    not_normalized = Or(
        IsNull("MsgType"),
        NotEqualTo("MsgType", FixMsg.into_instrument_msg_type()),
    )
    row_filter = not_normalized if window is None else And(window, not_normalized)
    reader = target.read_arrow_reader(
        field, row_filter=row_filter, order_by=("unix", "MsgSeqNum", "hash")
    )
    return FixMsg.from_arrow_reader(reader)


def _instrument_messages(versions):
    global instrument_versions
    for instrument in versions:
        if lower <= instrument.unix < upper:
            instrument_versions += 1
            yield instrument.into_fixmsg()


instrument_versions = instrument_written = 0
if lower is not None and upper is not None:
    versions = Instrument.from_fixmsgs(
        _market_messages(),
        registry=registry,
        instruments=_instrument_seeds(),
        snapshot_every=instrument_snapshot_every,
        snapshot_until=upper,
    )
    reader = FixMsg.into_arrow_reader(
        _instrument_messages(versions), batch_row_size=batch_row_size
    )
    instrument_written = target.append_arrow_reader(
        reader,
        field,
        merge_by=merge_by,
        commit_row_size=commit_row_size,
    )

result = {
    "read": read,
    "written": written + instrument_written,
    "skipped": skipped + instrument_versions - instrument_written,
    "raw_written": written,
    "instrument_versions": instrument_versions,
    "instrument_written": instrument_written,
    "routed": routed,
    "targets": {category: value.name for category, value in targets.items()},
}
try:
    import scrapbook as sb
except ImportError:
    pass
else:
    sb.glue("result", result, encoder="json")
result